# ODSC 2026: Deploying a Secure Autonomous Agent

We're building an agent that autonomously monitors teacher licensure compliance requirements across US states and districts — detecting changes and alerting you the moment something updates.

| Part | What you build | Complexity |
| --- | --- | --- |
| **Part 1** | Environment setup and first compliance check | Setup |
| **Part 2** | Custom Claw — 150 lines of Python, full monitor | Build it yourself |
| **Part 3** | Extend Custom Claw with Brave Search *(optional)* | Make it autonomous |
| **Part 4** | Add Telegram notifications to Custom Claw | Extend it |
| **Part 5** | Switch to OpenClaw — same logic, production framework | Use a framework |
| **Part 6** | Observability with Opik — trace every agent decision | Production ready |

**Prerequisites:** Docker running, `.env` filled in from `example.env`.

---
## Part 1: Environment Setup

In [ ]:
import subprocess
import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
BRAVE_API_KEY  = os.getenv("BRAVE_API_KEY", "")

assert OPENAI_API_KEY and not OPENAI_API_KEY.startswith("your-"), "Set OPENAI_API_KEY in .env"

if not BRAVE_API_KEY or BRAVE_API_KEY.startswith("your-"):
    print("⚠️  BRAVE_API_KEY not set — Part 3 (Brave Search extension) will be skipped")
    BRAVE_API_KEY = ""
else:
    print("✅ Brave key set")

docker_ok = subprocess.run("docker info".split(), capture_output=True).returncode == 0
assert docker_ok, "Docker is not running. Start Docker Desktop and try again."

print("✅ OpenAI key set")
print("✅ Docker running")
print("\nReady to go.")

---
## Part 2: Custom Claw — Build It Yourself

Before we bring in a framework, let's see the core logic: ~150 lines of Python that fetches state DOE pages, hashes the content, and detects changes.

Read `custom-claw/src/monitor.py` — that's the entire agent.

### What it does
1. Fetches each URL in `compliance_urls.py`
2. Strips HTML, hashes the text content with SHA-256
3. Compares against the stored hash in SQLite
4. Records a change if the hash differs
5. Sends a Telegram alert (Part 3)

In [ ]:
import shutil

# Auto-detect Docker sudo requirement
SUDO = ""
result = subprocess.run("docker info".split(), capture_output=True)
if result.returncode != 0:
    result = subprocess.run("sudo docker info".split(), capture_output=True)
    if result.returncode == 0:
        SUDO = "sudo "

DOCKER_COMPOSE = (
    f"{SUDO}docker compose"
    if subprocess.run(f"{SUDO}docker compose version".split(), capture_output=True).returncode == 0
    else f"{SUDO}docker-compose"
)

# Sync .env into custom-claw directory
shutil.copy(".env", "custom-claw/.env")

print(f"Using: {DOCKER_COMPOSE}")
print("Building Custom Claw container...")
!{DOCKER_COMPOSE} -f custom-claw/docker-compose.yml build
print("\n✅ Custom Claw built.")

In [ ]:
# Run the first compliance check — this captures baseline snapshots
print("Running first compliance check (capturing baselines)...\n")
!{DOCKER_COMPOSE} -f custom-claw/docker-compose.yml run --rm custom-claw

In [ ]:
# Check a specific state
!{DOCKER_COMPOSE} -f custom-claw/docker-compose.yml run --rm custom-claw python monitor.py Ohio

In [ ]:
# View the change log (SQLite)
import sqlite3
import os

DB_PATH = "data/compliance.db"
if os.path.exists(DB_PATH):
    conn = sqlite3.connect(DB_PATH)
    
    print("=== Snapshots ===")
    rows = conn.execute("SELECT state, subject, checked_at FROM snapshots ORDER BY checked_at DESC LIMIT 15").fetchall()
    for r in rows:
        print(f"  {r[2][:16]}  {r[0]} — {r[1]}")
    
    print("\n=== Changes Detected ===")
    rows = conn.execute("SELECT state, subject, url, detected_at FROM changes ORDER BY detected_at DESC").fetchall()
    if rows:
        for r in rows:
            print(f"  🚨 {r[3][:16]}  {r[0]} — {r[1]}")
            print(f"     {r[2]}")
    else:
        print("  No changes yet (baselines just captured).")
    
    conn.close()
else:
    print("Database not found — run the compliance check first (cell above).")

---
## Part 3: Extending Custom Claw with Brave Search *(Optional)*

**Skip this part if you don't have a Brave API key** — free at [brave.com/search/api](https://brave.com/search/api/) but not required for the rest of the workshop.

### The limitation we're solving

Custom Claw's watch list is hardcoded in `compliance_urls.py`. If a state adds a new page, or we want to monitor a district we've never seen before, we have to edit the code.

What if the agent could *find* the right URLs itself?

### The idea

Add a `discover_urls(state)` function that:
1. Queries Brave Search for `"{state} teacher licensure certification requirements site:.gov"`
2. Returns the top results as candidate URLs
3. Feeds them into the existing hash/diff/alert flow

This is exactly what OpenClaw's `brave-search` skill does automatically when you ask it to monitor a new state — no code changes needed. We're building a manual version of that here to show the concept.

In [ ]:
import requests

BRAVE_API_KEY = os.getenv("BRAVE_API_KEY", "")

def discover_compliance_urls(state: str, max_results: int = 3) -> list[dict]:
    if not BRAVE_API_KEY:
        print("⚠️  BRAVE_API_KEY not set — skipping URL discovery")
        return []

    query = f"{state} teacher licensure certification requirements site:.gov"
    resp = requests.get(
        "https://api.search.brave.com/res/v1/web/search",
        headers={"Accept": "application/json", "X-Subscription-Token": BRAVE_API_KEY},
        params={"q": query, "count": max_results},
        timeout=10,
    )
    resp.raise_for_status()

    results = resp.json().get("web", {}).get("results", [])
    return [{"title": r["title"], "url": r["url"]} for r in results]


# --- Demo ---
state = "Indiana"
print(f"Discovering compliance URLs for {state}...\n")
discovered = discover_compliance_urls(state)

if discovered:
    for r in discovered:
        print(f"  {r['title']}")
        print(f"  {r['url']}\n")
    print("These URLs could be passed directly into Custom Claw's run_check() —")
    print("no hardcoded watch list needed.")
else:
    print("(Brave API not configured — in the full setup, results would appear here.)\n")
    print("With OpenClaw, you'd just say:")
    print('  \"Check for compliance updates in Indiana\"')
    print("...and it searches, finds the pages, monitors them, and alerts you automatically.")

---
### Part 3b: Minimum Viable Conversational Agent *(Optional)*

We added Brave Search, but it's still structured code — you have to call `discover_compliance_urls("Indiana")` yourself.

What if you could just *ask* it?

Below is a ~50-line conversational wrapper around the functions you've already built. It uses the OpenAI API to interpret a natural language instruction, extract intent, and call the right function. This is the minimum viable version of what OpenClaw does under the hood.

> **Notice what you have to build yourself:** the system prompt (your mini `AGENTS.md`), the intent parser, the tool routing, the response formatter. OpenClaw handles all of this for you — the gap between this cell and Part 5 is everything a production agent framework provides.

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """You are a teacher licensure compliance monitoring agent.

You have two capabilities:
1. check_compliance(states: list[str]) — check monitored URLs for changes in specific states
2. discover_urls(state: str) — search for new compliance pages for a state

When the user gives you an instruction, respond with a JSON object like:
  {"action": "check_compliance", "states": ["Ohio", "Texas"]}
  {"action": "discover_urls", "state": "Indiana"}
  {"action": "unknown", "message": "I can only check compliance or discover URLs"}

Respond with JSON only. No explanation."""

def parse_intent(user_message: str) -> dict:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)

def run_agent(user_message: str):
    print(f"You: {user_message}")
    intent = parse_intent(user_message)
    action = intent.get("action")

    if action == "check_compliance":
        states = intent.get("states", [])
        print(f"Agent: Running compliance check for {', '.join(states)}...\n")
        # Import here so this cell is self-contained
        import sys; sys.path.insert(0, "custom-claw/src")
        from monitor import run_check
        changes = run_check(states)
        if changes:
            print(f"Agent: Found {len(changes)} change(s):")
            for c in changes:
                print(f"  🚨 {c['state']} — {c['subject']}: {c['url']}")
        else:
            print("Agent: No changes detected.")

    elif action == "discover_urls":
        state = intent.get("state", "")
        print(f"Agent: Searching for compliance pages for {state}...\n")
        results = discover_compliance_urls(state)
        if results:
            print(f"Agent: Found {len(results)} page(s) for {state}:")
            for r in results:
                print(f"  • {r['title']}")
                print(f"    {r['url']}")
        else:
            print("Agent: No results found (Brave API not configured).")

    else:
        print(f"Agent: {intent.get('message', 'I can check compliance or discover URLs for a state.')}")
    print()


# --- Try it ---
run_agent("Check for any licensure changes in Ohio and Texas")
run_agent("Find compliance pages for Indiana")
run_agent("What's the weather like?")

---
## Part 4: Telegram Notifications

Add `TELEGRAM_BOT_TOKEN` and `TELEGRAM_CHAT_ID` to your `.env` to receive alerts on your phone when compliance changes are detected.

See the [Telegram Setup](#telegram-setup) section in the README for how to create a bot and get your chat ID.

Once set, re-run the check from Part 2 — any detected change will be sent as a Telegram message.

In [ ]:
load_dotenv(override=True)

BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
CHAT_ID   = os.getenv("TELEGRAM_CHAT_ID", "")

if not BOT_TOKEN or not CHAT_ID:
    print("⚠️  TELEGRAM_BOT_TOKEN or TELEGRAM_CHAT_ID not set in .env")
    print("   Follow the Telegram setup in README.md, then re-run this cell.")
else:
    import requests
    resp = requests.post(
        f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
        json={"chat_id": CHAT_ID, "text": "✅ Compliance monitor connected. You'll receive alerts here when licensure requirements change."},
        timeout=10
    )
    if resp.ok:
        print("✅ Telegram connected — check your phone!")
    else:
        print(f"❌ Telegram error: {resp.text}")

---
## Part 5: Switch to OpenClaw

Custom Claw works, but it's a single Python script with no scheduling, no dashboard, no memory, and no multi-channel support. 

[OpenClaw](https://github.com/openclaw/openclaw) is a production agent framework that wraps the same logic with:
- Persistent memory and conversation context
- Web dashboard for monitoring
- Built-in scheduling
- Telegram, Slack, Discord, WhatsApp out of the box
- 50+ built-in skills
- Skills defined as Markdown — no Python required

The `check-compliance-updates` skill in `openclaw-agent/skills/` is the OpenClaw equivalent of `monitor.py`.

### Building `AGENTS.md` — Your Agent's Domain Knowledge

With Custom Claw, domain knowledge lives in the code: hardcoded URLs, hardcoded logic, hardcoded subjects.

With an LLM-powered agent, you externalize that knowledge into `AGENTS.md` — a plain text file the agent reads as context. It tells the agent:
- **What it does** — its purpose and scope
- **Domain knowledge** — what it needs to know about teacher licensure, Praxis exams, key sources
- **Skills available** — what tools it can call
- **Behavior guidelines** — how to respond, what to cite, when to search

Open `AGENTS.md` in the project root. That file is what gives OpenClaw its expertise.

> **The pattern:** Any time you build an LLM agent, start here. Define the domain, the tools, and the behavioral guardrails *before* you write a line of code. The agent reasons from this context — this is where the intelligence lives.

In [ ]:
import shutil
import time

OPENCLAW_COMPOSE = f"{DOCKER_COMPOSE} -f openclaw-agent/docker-compose.yml"

# Sync .env
shutil.copy(".env", "openclaw-agent/.env")

print("Starting OpenClaw gateway...")
!{OPENCLAW_COMPOSE} up -d openclaw-gateway

for i in range(15):
    health = subprocess.run(
        f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
        capture_output=True
    )
    if health.returncode == 0:
        print("✅ Gateway healthy!")
        break
    time.sleep(3)
    print(f"   Waiting... ({i+1})")

In [ ]:
# Configure the agent
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set agents.defaults.model openai/gpt-4o
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set gateway.bind lan
print("\n✅ Agent configured.")

In [ ]:
# Ask the agent to run a compliance check
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs agent -m "Check for any teacher licensure compliance updates in Ohio and Texas."

In [ ]:
# Connect Telegram to OpenClaw (get pairing code)
print("Adding Telegram channel to OpenClaw...")

TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
assert TELEGRAM_BOT_TOKEN, "Set TELEGRAM_BOT_TOKEN in .env first"

!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs channels add --channel telegram --token {TELEGRAM_BOT_TOKEN}

print("\n✅ Telegram channel added.")
print("Send the pairing code shown above to your bot on Telegram to complete setup.")

---
## Part 6: Observability with Opik

OpenClaw is now running, but we have no visibility into *how* it's making decisions — which tools it called, how long they took, what it cost.

Run `opik_observability.ipynb` to install the Opik plugin and start capturing full traces for every agent turn.

Then run `opik_evaluation.ipynb` to batch-evaluate your agent's response quality with LLM-as-a-judge metrics.